# Aedes container detector: training
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Needs `MyDrive/aedes/aedes_raw.zip` (contains `prepare_dataset.py` + `data/raw/`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/aedes'
!mkdir -p /content/work && cp $DRIVE/aedes_raw.zip /content/work/ && cd /content/work && unzip -qo aedes_raw.zip -x '__MACOSX/*' && ls

In [ ]:
!pip install -q ultralytics

In [ ]:
# leak-free re-split (prints counts + asserts no photo overlap)
%cd /content/work
!python prepare_dataset.py --raw data/raw --out /content/split
# extra yaml so we can evaluate the held-out new-area set
y = open('/content/split/data.yaml').read().replace('test: test/images', 'test: new_area/images')
open('/content/split/new_area.yaml', 'w').write(y)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data='/content/split/data.yaml',
    imgsz=640, epochs=80, patience=20, batch=16,
    degrees=10, fliplr=0.5, flipud=0.5,
    project=f'{DRIVE}/runs', name='yolov8s_v1', exist_ok=True,
)
BEST = str(model.trainer.best)
print('best weights:', BEST)

In [ ]:
# ONLY if Colab disconnected mid-training: re-run the drive + unzip + split cells, then this one
# from ultralytics import YOLO
# DRIVE = '/content/drive/MyDrive/aedes'
# model = YOLO(f'{DRIVE}/runs/yolov8s_v1/weights/last.pt')
# model.train(resume=True)
# BEST = str(model.trainer.best)

In [ ]:
import pandas as pd
best = YOLO(BEST)
def report(yaml, label):
    m = best.val(data=yaml, split='test', imgsz=640, plots=True,
                 project=f'{DRIVE}/runs', name=f'eval_{label}', exist_ok=True)
    rows = [{'class': best.names[c], 'P': m.box.p[i], 'R': m.box.r[i], 'mAP50': m.box.ap50[i], 'mAP50-95': m.box.ap[i]}
            for i, c in enumerate(m.box.ap_class_index)]
    rows.append({'class': 'ALL', 'P': m.box.mp, 'R': m.box.mr, 'mAP50': m.box.map50, 'mAP50-95': m.box.map})
    df = pd.DataFrame(rows).round(3); print(f'\n== {label} =='); print(df.to_string(index=False))
    df.to_csv(f'{DRIVE}/runs/results_{label}.csv', index=False)
report('/content/split/data.yaml', 'test')
report('/content/split/new_area.yaml', 'new_area')

In [ ]:
from IPython.display import Image, display
for label in ['test', 'new_area']:
    print(label); display(Image(f'{DRIVE}/runs/eval_{label}/confusion_matrix.png', width=700))

In [ ]:
# predictions on 6 new-area tiles (unseen location)
import glob
imgs = sorted(glob.glob('/content/split/new_area/images/*'))[:6]
for r in best.predict(imgs, conf=0.25, imgsz=640):
    display(Image(data=__import__('cv2').imencode('.jpg', r.plot())[1].tobytes(), width=450))

In [ ]:
!cp {BEST} {DRIVE}/best.pt && ls -la {DRIVE}